# PatchTST — walk-forward training across six months (Google Colab)

Trains the **side model** for [BipowerQuant](https://github.com/NynsenFaber/BipowerQuant):
given a 5-minute lookback, which triple barrier does the price touch first?

Three things differ from the earlier single-month notebook.

1. **Six months, walk-forward.** One model per fold, each trained only on months
   *before* its test month, so every number is a forecast rather than an in-sample
   fit. Three folds by default.
2. **A time budget.** Cell 6 measures real training throughput, projects the cost
   of the whole run, and picks the training stride that fits `TIME_BUDGET_HOURS`.
   It will not let you start an eleven-hour job by accident.
3. **It exports probabilities, not just weights.** The last cell writes one array
   per fold, which `python/walkforward.py --patchtst-probs` folds into exactly the
   backtest the tabular models go through — same execution model, same costs.

### Before you press Run all

1. `Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**`
2. Run all. It pauses once, near the start, for Google Drive authorisation.

Expect **3-6 hours**. Bar caches are written to Drive, so a disconnect costs you
the training but never the 50 GB of downloads.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU attached — Runtime > Change runtime type > T4 GPU"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device:", props.name, f"| {props.total_memory / 1e9:.1f} GB",
          "| native bf16:", torch.cuda.get_device_capability(0)[0] >= 8)
    # Free, accuracy-neutral speed on Ampere and newer; a no-op on a T4.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


## 1. Pull the code

The branch below has to exist **on the remote** — push it before running this.

*No remote access?* Skip this cell, upload `sequence_matrix.py`, `patchtst_model.py`,
`patchtst_train.py`, `patchtst_folds.py`, `backtest.py`, `walkforward.py` and
`data_feeder.py` through the file browser into `/content/`, then run
`sys.path.insert(0, "/content")` instead.

In [ ]:
REPO_URL = "https://github.com/NynsenFaber/BipowerQuant.git"
BRANCH   = "add-patchTST"
REPO_DIR = "/content/BipowerQuant"

import os, sys, subprocess

%pip install -q polars

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

sys.path.insert(0, f"{REPO_DIR}/python")

import glob, json, shutil, time, urllib.request, zipfile
import numpy as np

import patchtst_folds as pf
import sequence_matrix as seq
import walkforward as wf
from patchtst_model import PatchTSTConfig
from patchtst_train import TrainConfig, pick_device

print("code loaded from", REPO_DIR)


## 2. Configuration

`MONTHS` drives everything downstream. Six months gives three anchored folds at
`TRAIN_MONTHS = 3`; two months is enough to check the pipeline runs.

`TIME_BUDGET_HOURS` is enforced, not advisory — cell 6 raises the training stride
until the projected run fits it.

In [ ]:
# ---- data ----
SYMBOL    = "BTCUSDT"
MONTHS    = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
USE_DRIVE = True                                   # cache bars + weights on Drive
DRIVE_DIR = "/content/drive/MyDrive/BipowerQuant"
DATA_DIR  = "/content/data"

# ---- problem definition (keep matched to python/sequence_matrix.py) ----
WINDOW   = 300          # 5-minute lookback = PatchTST's sequence length L
HORIZON  = 3600         # vertical barrier: decide within one hour
BARRIER  = 0.0060       # +/-60 bp horizontal barriers
CHANNELS = "raw"        # "raw" = log_return + ofi; "full" = the old 6 channels

# ---- walk-forward ----
SCHEME       = "anchored"     # "anchored" | "rolling" | "holdout"
TRAIN_MONTHS = 3

# ---- model ----
PATCH_LEN, PATCH_STRIDE = 16, 8
D_MODEL, N_HEADS, N_LAYERS, D_FF = 64, 4, 6, 128
DROPOUT, HEAD_DROPOUT = 0.2, 0.2
USE_SCALE_FEATURES = True
NORM = "batch"

# ---- training ----
EPOCHS, BATCH_SIZE = 12, 512
LR, WEIGHT_DECAY   = 3e-4, 1e-4
PATIENCE           = 3
SEED               = 42
VAL_FRAC           = 0.10     # carved off the END of each fold's training months
VAL_STRIDE         = 8

# ---- the budget that keeps this an overnight job ----
TIME_BUDGET_HOURS = 6.0


## 3. Fetch the tape

Each month is downloaded, folded into 1-second bars, cached to Drive as a ~27 MB
`.npz`, and the multi-gigabyte CSV is **deleted immediately**. Six months of raw
CSV is ~52 GB and does not fit on a Colab disk; six months of bars is 165 MB.

Already-cached months are skipped, so a reconnect costs nothing.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
CACHE_DIR = DRIVE_DIR if USE_DRIVE else DATA_DIR

bar_paths = []
for month in MONTHS:
    cache = f"{CACHE_DIR}/bars_{SYMBOL}_{month}.npz"
    bar_paths.append(cache)
    if os.path.exists(cache):
        print(f"{month}: cached")
        continue

    url = (f"https://data.binance.vision/data/spot/monthly/trades/"
           f"{SYMBOL}/{SYMBOL}-trades-{month}.zip")
    zip_path = f"{DATA_DIR}/{SYMBOL}-trades-{month}.zip"
    started = time.perf_counter()
    print(f"{month}: downloading ...", end=" ", flush=True)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = archive.namelist()[0]
        archive.extract(csv_name, DATA_DIR)
    os.remove(zip_path)

    print("folding to bars ...", end=" ", flush=True)
    month_bars = seq.load_second_bars(f"{DATA_DIR}/{csv_name}", hours=None)
    seq.save_bars(month_bars, cache)
    os.remove(f"{DATA_DIR}/{csv_name}")   # never hold two months of CSV at once
    del month_bars
    print(f"done in {time.perf_counter() - started:.0f}s")

print(f"\n{len(bar_paths)} month(s) ready")


## 4. Splice and label

The months are spliced into one continuous 1-second grid — the market does not
restart between archives — and labelled with the triple barrier.

In [ ]:
bars = seq.load_bar_caches(bar_paths)
meta = bars["meta"]
print(f"{meta['n_bars']:,} bars | {(meta['last_ts'] - meta['first_ts']) / 86400:.0f} days "
      f"| {meta['empty_seconds'] / meta['n_bars']:.1%} trade-less seconds")

starts  = seq.valid_window_starts(bars["price"].size, WINDOW, HORIZON)
side, _, defined = seq.triple_barrier(bars["price"], HORIZON, BARRIER)
label_bar = starts + WINDOW - 1
touched = (side[label_bar] != 0) & defined[label_bar]

print(f"{starts.size:,} windows | barrier touched in {touched.mean():.2%} "
      f"| upper-first among those {(side[label_bar][touched] > 0).mean():.2%}")


## 5. Folds

One model per fold, each trained only on months *before* its test month. Nothing
is carried between folds — reusing weights would leak a later month into an
earlier month's forecast.

Training keeps only windows where a barrier is actually touched, since a window
with no side has nothing to teach a side model. **Test scores every window**,
resolved or not: whether a window is worth trading is the *gate's* decision, taken
downstream in `walkforward.py`, and the gate needs a side for anything it lets
through.

In [ ]:
device = pick_device()
purge  = WINDOW + HORIZON - 1
folds  = wf.build_folds(bars["ts"], SCHEME, TRAIN_MONTHS)

print(f"device: {device}\n")
print(f"{'fold':<26} {'train':>12} {'val':>10} {'test (all)':>12}")
for f in folds:
    tr, va, te = pf.fold_rows(f, starts, touched, purge, VAL_FRAC)
    print(f"{f.name:<26} {tr.size:>12,} {va.size:>10,} {te.size:>12,}")


## 6. Train, inside the time budget

`train_folds` runs a throughput probe first — the *real* training step, same
autocast and gradient clipping, on a throwaway copy of the model — then picks the
smallest stride whose projected wall time across every fold fits
`TIME_BUDGET_HOURS`, and only then starts training.

Windows overlap by 299 of 300 bars, so striding discards far less information than
it discards rows, which is what makes this safe to decide automatically. The
projection ignores early stopping, so it is an upper bound.

Pass `train_stride=N` instead of `time_budget_hours` to fix it by hand.

In [ ]:
probabilities, run_meta = pf.train_folds(
    bars, folds,
    window=WINDOW, horizon=HORIZON, barrier=BARRIER, channel_set=CHANNELS,
    config=PatchTSTConfig(
        n_channels=len(seq.CHANNEL_SETS[CHANNELS]), seq_len=WINDOW,
        patch_len=PATCH_LEN, stride=PATCH_STRIDE,
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
        dropout=DROPOUT, head_dropout=HEAD_DROPOUT,
        use_scale_features=USE_SCALE_FEATURES, norm=NORM,
    ),
    train_config=TrainConfig(
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
        patience=PATIENCE, seed=SEED, amp=True,
    ),
    val_stride=VAL_STRIDE, val_frac=VAL_FRAC,
    time_budget_hours=TIME_BUDGET_HOURS,
    device=device, checkpoint_dir="/content",
)

print(f"\n{'fold':<26} {'test ROC-AUC':>13} {'best epoch':>11}")
for name, row in run_meta["folds"].items():
    print(f"{name:<26} {row['roc_auc']:>13.4f} {row['best_epoch']:>11}")


## 7. Export

`patchtst_probs.npz` holds one probability array per fold, keyed by the fold name
`walkforward.build_folds` generates. Download it, put it beside your bar caches,
and the local backtest scores PatchTST through the identical execution model the
tabular models go through:

```bash
cd python
python walkforward.py --bars ../data/bars_6m.npz \\
                      --patchtst-probs ../data/patchtst_probs.npz
```

Read the ROC-AUC above as a *diagnostic*, not the result. It is measured only on
the windows where a barrier resolved, which is not a population you can select
into at decision time. The number that means something is the Sharpe the backtest
reports, on every window the gate lets through.

In [ ]:
out = "/content/patchtst_probs.npz"
np.savez_compressed(out, **probabilities)
with open("/content/patchtst_folds.json", "w") as fh:
    json.dump(run_meta, fh, indent=2, default=float)
print(f"{out}  ({os.path.getsize(out) / 1e6:.1f} MB)")

if USE_DRIVE:
    for name in ("patchtst_probs.npz", "patchtst_folds.json"):
        shutil.copy(f"/content/{name}", f"{DRIVE_DIR}/{name}")
    for ckpt in glob.glob("/content/patchtst_*.pt"):
        shutil.copy(ckpt, DRIVE_DIR)
    print(f"copied to {DRIVE_DIR}")

from google.colab import files
files.download(out)
